# DriftMind Cold-Start Forecasting Demo

This notebook shows how to:

1. Load DriftMind credentials from the environment or a `.env` file.
2. Create a `DriftMindClient` and a new forecaster.
3. Generate synthetic sinusoidal data with drifts.
4. Feed data online and request forecasts.
5. Visualize actual vs predicted values and global anomaly signals.

In [ ]:
from __future__ import annotations

import pandas as pd

from driftmind import DriftMindClient, generate_sin_cos_tan_with_drifts
from driftmind.exceptions import DriftMindConfigError, DriftMindError
from driftmind.utils import (
    load_credentials,
    plot_actual_vs_predicted,
    plot_time_series,
)

## 1. Load DriftMind credentials

Credentials are expected in the environment as:

- `DRIFTMIND_API_KEY`
- `DRIFTMIND_API_URL`

Optionally, they can be loaded from a `.env` file for local runs.

In [ ]:
try:
    creds = load_credentials()
    api_key = creds["DRIFTMIND_API_KEY"]
    base_url = creds["DRIFTMIND_API_URL"]
    print("✅ Credentials loaded.")
except DriftMindConfigError as err:
    raise RuntimeError(
        "Failed to load DriftMind credentials. "
        "Ensure DRIFTMIND_API_KEY and DRIFTMIND_API_URL are set in "
        "your environment or .env file."
    ) from err

# Create client (no context manager in notebooks for convenience)
client = DriftMindClient(api_key=api_key, base_url=base_url)
print(f"✅ Client created: {client}")

## 2. Create a new forecaster

We create a fresh forecaster for this demo run.

In [ ]:
columns = ["sin", "cos", "tan"]

forecaster_payload = {
    "forecaster_name": "Cold Start Demo",
    "features": columns,
    "input_size": 15,
    "output_size": 1,
}

try:
    forecaster_info = client.create_forecaster(forecaster_payload)
    forecaster_id = forecaster_info["forecaster_id"]
    print(f"✅ Created forecaster: {forecaster_id}")
    print(f"   Name: {forecaster_info['forecaster_name']}")
    print(f"   Features: {forecaster_info['features']}")
except DriftMindError as err:
    raise RuntimeError("Failed to create forecaster.") from err

## 3. Generate synthetic dataset with drifts

We now generate a dataset with sinusoidal drifts using a helper from `driftmind.generator`.

It creates a pandas DataFrame with columns:
- `sequence`: Integer time index from 0 to n-1.
- `sin`: Sine component with drifts.
- `cos`: Cosine component with drifts.
- `tan`: Clipped tangent component with drifts.

In [ ]:
df = generate_sin_cos_tan_with_drifts(n=600, noise_std=0.05, seed=42)
df.head()

## 4. Online loop: feed points and request forecasts

We iterate over the dataset, feed one point at a time, and request a forecast. We collect:

- Per-feature expected vs predicted values.
- Global anomaly score and number of clusters.

In [ ]:
results: dict[str, list[dict[str, float]]] = {col: [] for col in columns}
global_results = {
    "timestamp": [],
    "anomaly_score": [],
    "number_of_clusters": [],
}

print("Starting online learning and forecasting...")

for _, row in df.iterrows():
    # Build payload for a single time step
    point = {col: [float(row[col])] for col in columns}

    try:
        client.feed_point(forecaster_id, point)
    except DriftMindError as err:
        print(f"❌ Failed to feed point: {err}")
        continue

    try:
        yhat = client.forecast(forecaster_id)
    except DriftMindError:
        # No forecast available yet; continue feeding
        continue

    if yhat is None:
        continue

    seq = int(row["sequence"])
    global_results["timestamp"].append(seq)
    global_results["anomaly_score"].append(float(yhat.get("anomaly_score", 0.0)))
    global_results["number_of_clusters"].append(int(yhat.get("number_of_clusters", 0)))

    # Use snake_case key 'features'
    features_map = yhat.get("features", {})

    for var in columns:
        feature = features_map.get(var, {})
        preds = feature.get("predictions", [])
        if not preds:
            continue

        pred_val = float(preds[0])
        exp_val = float(row[var])
        sme = abs(exp_val - pred_val)

        results[var].append(
            {
                "expected": exp_val,
                "predicted": pred_val,
                "timestamp": seq,
            }
        )

        if seq % 50 == 0:  # Print every 50 points
            print(
                f"Fed sequence={seq}, "
                f"Expected {var}={exp_val:.3f}, "
                f"Predicted {pred_val:.3f}, "
                f"SME={sme:.3f}"
            )

print(f"\n✅ Completed {len(df)} iterations")

## 5. Visualize actual vs predicted

We convert the collected results into DataFrames and use the plotting
helpers from `driftmind.utils`.

In [ ]:
for var in columns:
    df_var = pd.DataFrame(results[var])
    if df_var.empty:
        print(f"No prediction data collected for {var}.")
        continue
    plot_actual_vs_predicted(df_var, variable_name=var)

## 6. Visualize global anomaly score and number of clusters

In [ ]:
df_global = pd.DataFrame(global_results)

if df_global.empty:
    print("No global metrics collected.")
else:
    plot_time_series(
        df_global["timestamp"],
        df_global["anomaly_score"],
        title="Global Anomaly Score over Time",
        xlabel="Time Step",
        ylabel="Anomaly Score",
    )

    plot_time_series(
        df_global["timestamp"],
        df_global["number_of_clusters"],
        title="Global Number of Clusters over Time",
        xlabel="Time Step",
        ylabel="Number of Clusters",
    )

## 7. Inspect forecaster details

In [ ]:
try:
    details = client.get_forecaster_details(forecaster_id)

    print("=== Forecaster Details ===")
    print(f"ID: {details['forecaster_id']}")
    print(f"Name: {details['forecaster_name']}")
    print("\nConfiguration:")
    print(f"  Input Size: {details['configuration']['input_size']}")
    print(f"  Output Size: {details['configuration']['output_size']}")

    print("\nFeature Statistics:")
    for name, stats in details["features"].items():
        print(f"  {name}:")
        print(f"    Active Clusters: {stats['active_clusters']}")
        print(f"    Total Observations: {stats['total_observations']}")
        print(f"    Anomaly Score: {stats['anomaly_score']}")

except DriftMindError as err:
    print(f"❌ Failed to fetch forecaster details: {err}")

## 8. Cleanup (optional)

Delete the forecaster when done. You can also close the client session.

In [ ]:
# Uncomment to delete the forecaster
# try:
#     client.delete_forecaster(forecaster_id)
#     print(f"✅ Deleted forecaster: {forecaster_id}")
# except DriftMindError as err:
#     print(f"❌ Failed to delete forecaster: {err}")

# Close the client session
client.close()
print("✅ Client session closed")